If you aggregate the prediction of a group of predictors (such as classifiers or regressors), you will often get better predictions than the best individual predictor. A group of predictors is called an `ensemble`; thus, this technique is called `ensemble learning`, and an ensemble learning algorithm is called `ensemble method`. 

# Voting Classifiers 

In [3]:
# A very simple way to create an even better classifier is to aggregate the predictions of each classifier: the class that gets the most votes is the
# ensemble's prediction. This majority-vote classifier is caalled a Hard Voting Classifier. 

from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC 

X, y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)), 
        ('rf', RandomForestClassifier(random_state=42)), 
        ('svc', SVC(random_state=42))
    ]
)
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression(random_state=42)),
                             ('rf', RandomForestClassifier(random_state=42)),
                             ('svc', SVC(random_state=42))])

In [5]:
for name, clf in voting_clf.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test))

lr = 0.864
rf = 0.896
svc = 0.896


In [7]:
voting_clf.score(X_test, y_test)

0.912

The voting classifier outperforms all the individual classifiers. If all classifiers are able to estimate class probabilities (i.e. if they all have a predict_proba() method), then you should generally tell Scikit-Learn to predict the class with the highest class probability, averaged over all the individual classifiers. This is called soft voting. 

In [9]:
voting_clf.voting = "soft"
voting_clf.named_estimators["svc"].probability = True 
voting_clf.fit(X_train, y_train)
voting_clf.score(X_test, y_test)

0.92

We reach 92% accuracy simply by using soft voting - not bad! 

# Bagging and Pasting

One way to get a diverse set of classifiers is to use very different training algorithms, as just discussed. Another way is to use the same training algorithm for every predictor but train them on different random subsets of the training set. When sampling is performed with replacement, this method is called `Bagging` (Short for bootstrap aggregating). When sampling is performed without replacement, it is called `Pasting`. 

In [15]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier 

bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=100,n_jobs=-1,random_state=42) # n_jobs = -1 means use all cpu cores
bag_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=100,
                  n_estimators=500, n_jobs=-1, random_state=42)

In [17]:
bag_clf.score(X_test, y_test)

0.904

## Out-of-Bag Evaluation

With bagging, some training instances may be sampled several times for any given predictor, while others may not be sampled at all. With this process, it can be shown mathematically that only about 63% of the training instances are sampled on average for each predictor. The remaining 37% of the training instances that are not sampled are called out-of-bag (OOB) instances. <br>
A bagging ensemble can be evaluated using OOB instances, without the need for a separate validation set: indeed, if there are enough estimators, then each instance in the training set will likely be an OOB instance of several estimators, so these estimators can be used to make a fair ensemble prediction for that instance. 

In [21]:
bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, oob_score=True, n_jobs=-1, random_state=42)

bag_clf.fit(X_train, y_train)
bag_clf.oob_score_

0.896

In [24]:
# According to this OOB evaluation, this BaggingClassifier is likely to achieve about 89.6% accuracy on the test set. Let's verify this: 
from sklearn.metrics import accuracy_score
y_pred = bag_clf.predict(X_test)
accuracy_score(y_test,y_pred)

0.92

# Random Forests

A random forest is an ensemble of decision trees, generally trained via the bagging method (or sometimes pasting), typically with max_samples set to the size of the training set. 

In [28]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1, random_state=42)

rnd_clf.fit(X_train, y_train) 

y_pred_rf = rnd_clf.predict(X_test)

In [30]:
accuracy_score(y_test, y_pred_rf)

0.912

In [36]:
# We can use the same using BaggingClassifier by taking max_features = square root of n (number of features). 

bag_clf = BaggingClassifier(DecisionTreeClassifier(max_features="sqrt", max_leaf_nodes=16), n_estimators=500, n_jobs=-1, random_state=42) 
bag_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(max_features='sqrt',
                                                   max_leaf_nodes=16),
                  n_estimators=500, n_jobs=-1, random_state=42)

In [38]:
y_pred_bagging = bag_clf.predict(X_test)

accuracy_score(y_test, y_pred_bagging)

0.912

## Feature Importance : 

In [43]:
# Yet another great quality of random forests is that they make it easy to measure the relative importance of each feature. 

from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
rnd_clf.fit(iris.data, iris.target)
for score, name in zip(rnd_clf.feature_importances_, iris.data.columns): 
    print(round(score, 2), name)

0.11 sepal length (cm)
0.02 sepal width (cm)
0.44 petal length (cm)
0.42 petal width (cm)


# Boosting

Boosting refers to any ensemble method that can combine several weak learners into a strong learner. The general idea of most boosting methods is to train predictors sequentially, each trying to correct its predecessor. 

## AdaBoost

One way for a new predictor to correct its predecessors is to pay a bit more attention to the training instances that the predecessor underfit. This result in a new predictiors focusing more and more on the hard cases. This is the technique used by AdaBoost (short form for Adaptive Boosting)

In [49]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=30, learning_rate=0.5, random_state=42, algorithm="SAMME")
ada_clf.fit(X_train, y_train)

AdaBoostClassifier(algorithm='SAMME',
                   estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=30, random_state=42)

In [52]:
y_pred_ada = ada_clf.predict(X_test)
accuracy_score(y_pred_ada, y_test)

0.88

## Gradient Boosting

Another very popular boosting algorithm is gradient boosting. just like AdaBoost, gradient boosting works by sequentially adding predictors to an ensemble, each one correcting its predecessor. However, instead of tweaking the instance weights at every iteration like AdaBoost does, this method tries to fit the new predictor to the residual errors made by the previous predictor.

In [56]:
import numpy as np 
from  sklearn.tree import DecisionTreeRegressor

m = 100 # number of instances 
rng = np.random.default_rng(seed=42)
X = rng.random((m,1)) - 0.5
noise = 0.05 * rng.standard_normal(m)
y = 3 * X[:, 0] ** 2 + noise # y = 3x^2 + Gaussian Noise 

tree_reg1 = DecisionTreeRegressor(max_depth=2, random_state=42) 
tree_reg1.fit(X, y) 

DecisionTreeRegressor(max_depth=2, random_state=42)

In [58]:
y2 = y - tree_reg1.predict(X) 
tree_reg2 = DecisionTreeRegressor(max_depth=2, random_state=43)
tree_reg2.fit(X, y2) 

DecisionTreeRegressor(max_depth=2, random_state=43)

In [60]:
y3 = y2 - tree_reg2.predict(X) 
tree_reg3 = DecisionTreeRegressor(max_depth=2, random_state=44) 
tree_reg3.fit(X, y3) 

DecisionTreeRegressor(max_depth=2, random_state=44)

In [62]:
X_new = np.array([[-0.4],[0.],[0.5]])
sum(tree.predict(X_new) for tree in (tree_reg1, tree_reg2, tree_reg3))

array([0.57356534, 0.0405142 , 0.66914249])

In [64]:
# Scikit-Learn has library GradientBoostingRegressor for this purpose 

from sklearn.ensemble import GradientBoostingRegressor

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbrt.fit(X, y)

GradientBoostingRegressor(learning_rate=1.0, max_depth=2, n_estimators=3,
                          random_state=42)

### Histogram-Based Gradient Boosting

Scikit-Learn also provides another GBRT(Gradient Boosted Regression Trees) implementation, optimized for large datasets : `Histogram-based Gradient Boosting (HGB)`. It works by binning the input features, replacing them with integers. 

In [74]:
import pandas as pd 

data = pd.read_csv("datasets/housing/housing.csv")
data.head()
housing = data.drop("median_house_value",axis=1)
housing_labels = data["median_house_value"].copy()

In [78]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder

hgb_reg = make_pipeline(
    make_column_transformer((OrdinalEncoder(), ["ocean_proximity"]), 
                           remainder="passthrough"), 
    HistGradientBoostingRegressor(categorical_features=[0], random_state=42)
)

hgb_reg.fit(housing, housing_labels)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ordinalencoder',
                                                  OrdinalEncoder(),
                                                  ['ocean_proximity'])])),
                ('histgradientboostingregressor',
                 HistGradientBoostingRegressor(categorical_features=[0],
                                               random_state=42))])

# Stacking

The last ensemble method in this chapter is called `stacking` (short for stacked generalization). It is based on a simple idea: instead of using trivial functions (such as hard voting) to aggregate the predictions of all predictors in an ensemble,  why don't we train a model to perform this aggregation? 

In [82]:
from sklearn.ensemble import StackingClassifier

stacking_clf = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)), 
        ('rf', RandomForestClassifier(random_state=42)), 
        ('svc', SVC(probability=True, random_state=42))
    ], 
    final_estimator=RandomForestClassifier(random_state=43), 
    cv=5 # number of cross validation folds 
)
stacking_clf.fit(X_train,  y_train)

StackingClassifier(cv=5,
                   estimators=[('lr', LogisticRegression(random_state=42)),
                               ('rf', RandomForestClassifier(random_state=42)),
                               ('svc', SVC(probability=True, random_state=42))],
                   final_estimator=RandomForestClassifier(random_state=43))

In [84]:
stacking_clf.score(X_test, y_test)

0.928